# AH2179 Final Project — Short-Term Traffic Flow Prediction

## Project idea

Trafikverket (TRV) has provided MCS/TDIS traffic data from a test bottleneck in Stockholm as a Proof of Concept (POC) for short-term traffic prediction.

The dataset contains 1-minute traffic observations from **28 sensors distributed across 8 portals**.

The objective of this project is to develop a machine-learning model that predicts future traffic flow in real time. The prediction is updated every minute and focuses on the **15-minute prediction horizon** required by the assignment.

The main prediction target is **FLOW**.

### Main research question

> **Does incorporating spatial information from other traffic sensors improve short-term flow prediction compared with using only the target sensor's historical flow?**

### Research questions

1. **Baseline:** How accurately can future 15-minute traffic flow be predicted using only the target sensor's historical flow?

2. **Spatial information:** Does incorporating information from other traffic sensors improve 15-minute flow prediction compared with using only the target sensor's historical flow?

3. **Spatial extent:** How does prediction accuracy change as information from an increasing spatial range of sensors is incorporated?

4. **Spatial variation:** Does the effect of spatial information on prediction accuracy vary between sensors and portals?

### Important: flow-only prediction

The target variable is **FLOW**.

`SPEED_MS_AVG` is used for exploratory analysis and diagnostics, but it is **not used as a predictor** in the main prediction models.

### Data separation

The training dataset is used for:

- exploratory data analysis
- feature engineering
- model training
- validation
- feature comparison
- model comparison
- hyperparameter tuning

The independent evaluation dataset is kept separate and is used only for the final evaluation of the selected model.


structure of the notebook:

TRAINING DATA
     │
     ├── data cleaning
     ├── EDA
     ├── historical FLOW features
     ├── temporal features
     ├── future 15-min target
     │
     ↓
BASELINE
own sensor history
     │
     ↓
DEVELOPMENT EVALUATION
evaluation_dataset.csv
     │
     ↓
SPATIAL EXPERIMENT
     │
     ├── own sensor
     ├── same portal
     ├── nearby portals
     ├── wider network
     └── all sensors
     │
     ↓
ALGORITHM COMPARISON
     │
     ↓
HYPERPARAMETER TUNING
     │
     ↓
FINAL MODEL
     │
     ↓
FINAL EVALUATION
final_evaluation_dataset.csv

## 1. Import librariers

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
HORIZON = 15 # GG: possibly change here for the 15, 30, and 60 minutes

## 2. Import data

In [ ]:
# project folder - MISSING ON GITHUB
DATA_DIR = Path('data/dataset_for_traffic_project_assignment')

# datasets 
TRAIN_PATH = DATA_DIR / 'training_dataset.csv'
DEV_EVAL_PATH = DATA_DIR / 'evaluation_dataset.csv' # GG: you used this df before but don't we have only 2 df?
FINAL_EVAL_PATH = DATA_DIR / 'final_evaluation_dataset.csv'

MODEL_PATH = Path('models') / 'best_traffic_model.keras'
Path('models').mkdir(exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
# import traning data 
train_raw = pd.read_csv(
    TRAIN_PATH,
    sep=";"
)

print("Training data shape:", train_raw.shape)

display(train_raw.head())

### 2.1 Morning peak

In [ ]:
# extract morning peak
train = train_raw.copy()

train["time_only"] = train["timestamp"].dt.time

morning_peak = train.loc[
    train["timestamp"].dt.strftime("%H:%M").between(
        "07:30",
        "08:30"
    )
].copy()

print("Morning peak observations:", morning_peak.shape[0])

display(morning_peak.head())

## 3. Data engineering

In [ ]:
# create timestamp

train["timestamp"] = pd.to_datetime(
    train["Date"].astype(str) + " " + train["Time"].astype(str),
    errors="coerce"
)

train["FLOW"] = pd.to_numeric(
    train["FLOW"],
    errors="coerce"
)

train["SPEED_MS_AVG"] = pd.to_numeric(
    train["SPEED_MS_AVG"],
    errors="coerce"
)

train = train.sort_values(
    ["DP_ID", "timestamp"]
).reset_index(drop=True)

print("Earliest observation:", train["timestamp"].min())
print("Latest observation:", train["timestamp"].max())

display(train.head())


In [ ]:
# check sensors and portal
number_sensors = train["DP_ID"].nunique()
number_portals = train["PORTAL"].nunique()

print("Number of sensors:", number_sensors)
print("Number of portals:", number_portals)

print("\nSensors per portal:")

sensors_per_portal = (
    train.groupby("PORTAL")["DP_ID"]
    .nunique()
    .sort_values()
)

display(sensors_per_portal)

In [ ]:
# sensor/portal table

sensor_portal = (
    train[["DP_ID", "PORTAL"]]
    .drop_duplicates()
    .sort_values(["PORTAL", "DP_ID"])
    .reset_index(drop=True)
)

display(sensor_portal)

### 2.1 Missing values

In [ ]:
train = train_raw.copy()

print("Missing FLOW:", train["FLOW"].isna().sum())
print("Missing SPEED:", train["SPEED_MS_AVG"].isna().sum())
print("Negative FLOW:", (train["FLOW"] < 0).sum())
print("Zero FLOW:", (train["FLOW"] == 0).sum())
print("Duplicate sensor-minute rows:",
      train.duplicated(["DP_ID", "timestamp"]).sum())

display(train.groupby("PORTAL")["FLOW"].agg(["count", "mean", "median", "std"]))
train = train_raw.copy()


In [ ]:
# Remove rows where FLOW is missing.
# Zero FLOW values are retained.

train = train.dropna(
    subset=["FLOW"]
).copy()

print("Rows after removing missing FLOW:", len(train))
print("Remaining missing FLOW:", train["FLOW"].isna().sum())

## 4. Exploratory Data Analysis

Before creating prediction models, the traffic-flow data are explored to understand differences between sensors.

The main questions are:

- Do sensors have similar flow distributions?
- Are some sensors systematically busier than others?
- Are there large differences between portals?
- Are there strong temporal patterns?

### 4.1 Flow distribution

In [ ]:
sensor_statistics = (
    train.groupby("DP_ID")["FLOW"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .sort_values("mean", ascending=False)
)

display(sensor_statistics)

In [ ]:
# flow distribution
plt.figure(figsize=(10, 5))

sns.histplot(
    train["FLOW"].dropna(),
    bins=50,
    kde=True
)

plt.xlabel("FLOW")
plt.ylabel("Number of observations")
plt.title("Distribution of traffic FLOW across all sensors")
plt.show()

In [ ]:
# morning av flow

morning_sensor_flow = (
    morning_peak.groupby("DP_ID")["FLOW"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(14, 5))

morning_sensor_flow.plot(
    kind="bar"
)

plt.xlabel("Sensor")
plt.ylabel("Average FLOW")
plt.title("Average morning-peak FLOW by sensor")
plt.xticks(rotation=90)
plt.show()

In [ ]:
# daily flow

train["minute_of_day"] = (
    train["timestamp"].dt.hour * 60
    + train["timestamp"].dt.minute
)

daily_flow = (
    train.groupby("minute_of_day")["FLOW"]
    .mean()
)

plt.figure(figsize=(12, 5))

plt.plot(
    daily_flow.index,
    daily_flow.values
)

plt.xlabel("Minute of day")
plt.ylabel("Average FLOW")
plt.title("Average traffic FLOW during the day")
plt.show()

### 4.2 Speed-flow relationship

`SPEED_MS_AVG` is part of the supplied dataset and can provide useful information about traffic conditions.

However, the main project predicts **FLOW only**.

Therefore, speed is used here for exploratory analysis and later diagnostics, but it will **not be included as a predictive feature**.

This allows us to investigate whether unusual congestion is associated with larger prediction errors without using speed to directly predict flow.

In [ ]:
sample = train.dropna(
    subset=["FLOW", "SPEED_MS_AVG"]
).sample(
    min(20000, train.dropna(
        subset=["FLOW", "SPEED_MS_AVG"]
    ).shape[0]),
    random_state=RANDOM_STATE
)

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=sample,
    x="SPEED_MS_AVG",
    y="FLOW",
    alpha=0.25
)

plt.xlabel("Average speed (m/s)")
plt.ylabel("FLOW")
plt.title("Relationship between speed and traffic flow")
plt.show()

### 4.3 Sensor correlation

The main research question asks whether information from other sensors can improve prediction.

Before building the models, we therefore investigate whether traffic flow at different sensors is related.

Correlation is used only as an exploratory tool. A high correlation does not automatically mean that one sensor should be used as a predictor of another.

In [ ]:
sensor_flow_wide = (
    train.pivot_table(
        index="timestamp",
        columns="DP_ID",
        values="FLOW",
        aggfunc="mean"
    )
)

sensor_correlation = sensor_flow_wide.corr()

plt.figure(figsize=(12, 10))

sns.heatmap(
    sensor_correlation,
    cmap="coolwarm",
    center=0
)

plt.title("FLOW correlation between sensors")
plt.xlabel("Sensor")
plt.ylabel("Sensor")
plt.show()

## 5. Model Development - Prediction
this section can be added after te model development

### 5.1 Check the 1-minute time structure

The prediction is updated every minute, so it is important to check whether the dataset contains observations at approximately 1-minute intervals.

We will check:

- how often timestamps occur,
- whether there are gaps,
- whether different sensors have different amounts of missing time.

We will keep the original 1-minute resolution if the data supports it, because the final prediction is also updated every minute.

In [ ]:
# Calculate the time difference between consecutive observations for each sensor.

time_gaps = (
    train.sort_values(["DP_ID", "timestamp"])
    .groupby("DP_ID")["timestamp"]
    .diff()
)

print("Most common time differences:")

display(
    time_gaps.value_counts().head(10)
)

In [ ]:
# Find gaps larger than 1 minute
# A normal observation should be approximately 1 minute after the previous observation for the same sensor.

large_gaps = (
    train.loc[
        time_gaps > pd.Timedelta(minutes=1),
        ["DP_ID", "timestamp"]
    ]
    .copy()
)

print("Number of gaps larger than 1 minute:", large_gaps.shape[0])

display(large_gaps.head(20))

# summary gap: Count the number of gaps larger than 1 minute for each sensor.

gaps_by_sensor = (
    train.assign(
        time_gap=time_gaps
    )
    .loc[lambda x: x["time_gap"] > pd.Timedelta(minutes=1)]
    .groupby("DP_ID")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="number_of_large_gaps")
)

display(gaps_by_sensor)

#### Check whether FLOW contains zeros

A FLOW value of zero does not automatically mean that the observation is wrong.

For example, zero flow could represent:

- very low traffic,
- a temporary traffic interruption,
- a sensor issue,
- or missing/invalid data represented as zero.

Therefore, zeros will be investigated rather than automatically removed.

In [ ]:
# Calculate the percentage of zero FLOW observations for each sensor.

zero_summary = (
    train.groupby("DP_ID")["FLOW"]
    .agg(
        total_observations="count",
        zero_observations=lambda x: (x == 0).sum()
    )
)

zero_summary["zero_percent"] = (
    zero_summary["zero_observations"]
    / zero_summary["total_observations"]
    * 100
)

display(
    zero_summary.sort_values(
        "zero_percent",
        ascending=False
    )
)

### 5.2 Prediction setup: 15-minute rolling prediction

The model will make a new prediction every minute.

For example:

- At 07:00 → predict traffic flow from 07:01 to 07:16
- At 07:01 → predict traffic flow from 07:02 to 07:17
- At 07:02 → predict traffic flow from 07:03 to 07:18

Therefore, the prediction window moves forward by one minute at every prediction time.

For each sensor and prediction time, we first create the 15 future 1-minute FLOW observations.

The exact aggregation of these 15 observations will be decided after checking how FLOW is defined in the dataset.

In [ ]:
# Sort the training data by sensor and time.

train = train.sort_values(
    ["DP_ID", "timestamp"]
).reset_index(drop=True)

display(
    train[
        [
            "timestamp",
            "DP_ID",
            "PORTAL",
            "FLOW"
        ]
    ].head()
)

In [ ]:
# create future Flow values

# For every sensor, shift FLOW backwards so that each row contains information about future observations. For example:

# FLOW_t_plus_1 = FLOW one minute into the future
# FLOW_t_plus_2 = FLOW two minutes into the future
# ...
# FLOW_t_plus_15 = FLOW fifteen minutes into the future


for i in range(1, HORIZON + 1):

    train[f"FLOW_t_plus_{i}"] = (
        train
        .groupby("DP_ID")["FLOW"]
        .shift(-i)
    )

display(
    train[
        [
            "timestamp",
            "DP_ID",
            "FLOW"
        ]
        + [f"FLOW_t_plus_{i}" for i in range(1, 6)]
    ].head(20)
)

This markdown section explains easily the concept of the 15 minutes (delete after)
### Why are we using `shift(-i)`?

Normally, a lag looks backwards:

- `shift(1)` → previous minute
- `shift(5)` → five minutes ago

Here we need the future because we are creating the value that the model will eventually learn to predict:

- `shift(-1)` → one minute ahead
- `shift(-15)` → fifteen minutes ahead

These future values will be used **only to construct the target**.

They must never be included among the model predictors.

In [ ]:
# Select one sensor so that we can inspect the rolling window manually.

example_sensor = train["DP_ID"].iloc[0] # change here the sensor we want to predict

example_window = (
    train[
        train["DP_ID"] == example_sensor
    ]
    [
        [
            "timestamp",
            "FLOW"
        ]
        + [f"FLOW_t_plus_{i}" for i in range(1, 6)]
    ]
    .head(10)
)

print("Example sensor:", example_sensor)

display(example_window)

### Temporary 15 minutes summary
investigation if it is better to use the mean or aggregate them 

In [ ]:
# List of the 15 future FLOW columns.

future_flow_columns = [
    f"FLOW_t_plus_{i}"
    for i in range(1, HORIZON + 1)
]

# Calculate both possible summaries.

train["future_15min_mean"] = (
    train[future_flow_columns]
    .mean(axis=1)
)

train["future_15min_sum"] = (
    train[future_flow_columns]
    .sum(axis=1)
)

display(
    train[
        [
            "timestamp",
            "DP_ID",
            "FLOW",
            "future_15min_mean",
            "future_15min_sum"
        ]
    ].head(20)
)

### Check the rolling prediction window

The two temporary variables above represent different things:

- `future_15min_mean` = average FLOW across the 15 future minutes
- `future_15min_sum` = total of the FLOW values across the 15 future minutes

We will not choose between them based on model performance.

The choice should instead follow the meaning of the FLOW variable and the assignment definition of the prediction target.

In [ ]:
# Define the target as the sum of FLOW during the next 15 minutes.

TARGET_COLUMN = "future_15min_sum" # change to mean if needed

train["TARGET"] = train[TARGET_COLUMN]

print("Target variable:", TARGET_COLUMN)

display(
    train[
        [
            "timestamp",
            "DP_ID",
            "FLOW",
            "TARGET"
        ]
    ].head(20)
)

In [ ]:
# Remove observations that do not have a complete 15-minute future window.

complete_future_window = train[future_flow_columns].notna().all(axis=1)

train = train[complete_future_window].copy()

print(
    "Rows with a complete future prediction window:",
    len(train)
)

### 5.3 Create temporal features
GG: here is where i started to get a little bit loss


Traffic changes depending on the time of day and day of the week.

We therefore create simple calendar features such as:

- hour,
- minute,
- day of week,
- whether the observation is during the morning peak.

These features describe the prediction time.

They do not use any future traffic information.

In [ ]:
# Create calendar variables from the prediction timestamp.

train["hour"] = train["timestamp"].dt.hour

train["minute"] = train["timestamp"].dt.minute

train["day_of_week"] = (train["timestamp"].dt.dayofweek)

train["day_of_month"] = (train["timestamp"].dt.day)

train["month"] = (train["timestamp"].dt.month)


# Monday = 0, Sunday = 6
train["is_weekend"] = (
    train["day_of_week"] >= 5
).astype(int)

display(
    train[
        [
            "timestamp",
            "hour",
            "minute",
            "day_of_week",
            "is_weekend"
        ]
    ].head()
)

### 5.4 Create Historical FLOW features

The baseline model will use the target sensor's own previous FLOW observations.

For example, at 07:30 the model can use:

- current FLOW,
- FLOW one minute ago,
- FLOW five minutes ago,
- FLOW fifteen minutes ago,
- FLOW thirty minutes ago,
- FLOW sixty minutes ago.

These are all values that would already be available in a real-time prediction situation.

No future FLOW values are used as predictors.

In [ ]:
# Create historical FLOW features for each sensor.

train = train.sort_values(
    ["DP_ID", "timestamp"]
).reset_index(drop=True)

lag_minutes = [
    1,
    5,
    15,
    30,
    60
]

for lag in lag_minutes:

    train[f"FLOW_lag_{lag}"] = (
        train
        .groupby("DP_ID")["FLOW"]
        .shift(lag)
    )

display(
    train[
        [
            "timestamp",
            "DP_ID",
            "FLOW",
            "FLOW_lag_1",
            "FLOW_lag_5",
            "FLOW_lag_15",
            "FLOW_lag_30",
            "FLOW_lag_60",
            "TARGET"
        ]
    ].head(20)
)

In [ ]:
# Create rolling averages using only previous observations.
# GG: this needs to change depending if we are aggrgating or using he average

# IMPORTANT: shift(1) is used before rolling so that the current/future observations are not included.

for window in [5, 15, 30]:

    train[f"FLOW_previous_{window}min_mean"] = (
        train
        .groupby("DP_ID")["FLOW"]
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

display(
    train[
        [
            "timestamp",
            "DP_ID",
            "FLOW",
            "FLOW_previous_5min_mean",
            "FLOW_previous_15min_mean",
            "FLOW_previous_30min_mean"
        ]
    ].head(30)
)

### 5.5 Answer the questions

GG: here we can decide if we simply want to focus on configuration A C/D only
#### Add spatial information

The main research question is whether information from other sensors improves prediction.

We will compare increasingly large spatial information sets.

__Configuration A — Own sensor__

Only historical FLOW from the target sensor.

__Configuration B — Same portal__

Historical FLOW from the target sensor and the other sensors belonging to the same portal.

__Configuration C — Nearby portals__

Historical FLOW from sensors at nearby portals.

__Configuration D — Wider network__

Historical FLOW from a wider group of portals.

__Configuration E — All sensors__

Historical FLOW from all 28 sensors.

The sensor number is not assumed to represent the same lane at different portals because lane correspondence is unknown.

#### 5.5.1 Define the Baseline model features

The baseline represents the situation where we only use information from the target sensor itself.

The baseline therefore contains:

- the target sensor's historical FLOW,
- historical rolling FLOW,
- simple time/calendar information.

It does **not** contain FLOW from other sensors.

This gives us a reference point for testing whether spatial information actually improves prediction.

In [ ]:
baseline_features = [
    "FLOW",
    "FLOW_lag_1",
    "FLOW_lag_5",
    "FLOW_lag_15",
    "FLOW_lag_30",
    "FLOW_lag_60",
    "FLOW_previous_5min_mean",
    "FLOW_previous_15min_mean",
    "FLOW_previous_30min_mean",
    "hour",
    "minute",
    "day_of_week",
    "is_weekend"
]

print("Number of baseline features:", len(baseline_features))

print("\nBaseline features:")
print(baseline_features)

#### 5.5.3 Create wide flow data

In [ ]:
# Convert FLOW into a wide format.

# Each sensor becomes a separate column. This allows us to use other sensors as predictors.

flow_wide = (
    train
    .pivot_table(
        index="timestamp",
        columns="DP_ID",
        values="FLOW",
        aggfunc="mean"
    )
)

flow_wide.columns = [
    f"FLOW_SENSOR_{sensor}"
    for sensor in flow_wide.columns
]

flow_wide = flow_wide.sort_index()

print("Wide FLOW dataset shape:", flow_wide.shape)

display(flow_wide.head())

#### 5.5.4 Create lagged spatial variables

In [ ]:
# Create a one-minute historical value for every sensor.

# We use lag 1 so that the spatial information is information that was already available at the prediction time.

spatial_lags = flow_wide.shift(1)

spatial_lags.columns = [
    column + "_lag_1"
    for column in spatial_lags.columns
]

display(spatial_lags.head())

In [ ]:
# Add the spatial lag variables to the main dataset.

train = train.merge(
    spatial_lags,
    left_on="timestamp",
    right_index=True,
    how="left"
)

print("Dataset shape after adding spatial features:", train.shape)

#### 5.5.5 Configuration B: Build sensor groups

In [ ]:
# Create a dictionary showing which sensors belong to each portal.

portal_sensors = (
    sensor_portal
    .groupby("PORTAL")["DP_ID"]
    .apply(list)
    .to_dict()
)

for portal, sensors in portal_sensors.items():

    print("Portal:", portal)
    print("Sensors:", sensors)
    print()

In [ ]:
# Create the same-portal spatial features.
#
# For each target sensor, we include the lagged FLOW of the other sensors belonging to the same portal.

def get_same_portal_features(target_sensor):

    target_portal = sensor_portal.loc[
        sensor_portal["DP_ID"] == target_sensor,
        "PORTAL"
    ].iloc[0]

    sensors = portal_sensors[target_portal]

    features = []

    for sensor in sensors:

        if sensor != target_sensor:

            column = f"FLOW_SENSOR_{sensor}_lag_1"

            if column in train.columns:
                features.append(column)

    return features


example_sensor = train["DP_ID"].iloc[0]

print("Example sensor:", example_sensor)
print(
    "Same-portal features:",
    get_same_portal_features(example_sensor)
)

#### 5.5.6 Configuration C/D: nearby/wider groups network

__Define the spatial extent__

The portal order should follow the physical layout of the Trafikverket study area.

Do not assume that the numerical value of `PORTAL` represents physical distance.

The following cell should therefore be filled using the portal layout provided with the project data/course material.

In [ ]:
# Check the portal labels currently present in the dataset.

portal_labels = sorted(
    train["PORTAL"].dropna().unique()
)

print("Portal labels in the dataset:")
print(portal_labels)

In [ ]:
# Replace this list with the physical order of the portals from the project study-area diagram.

portal_order = portal_labels.copy()

print("Current portal order:")
print(portal_order)

#### 5.5.7 Chronological train-validation split

Because this is a real-time prediction problem, we should not randomly shuffle the observations.

The model should learn from the past and be tested on later observations.

Therefore:

- earlier 2021 data → training
- later 2021 data → validation
- 2022 → completely untouched final evaluation

The 2022 evaluation dataset will not be used for model selection or tuning.\
GG: i believe there might be an eaier way to do this like we have done in the labs

In [ ]:
# Sort the training data chronologically.

development_data = train.sort_values(
    "timestamp"
).reset_index(drop=True)

print("Development data:", development_data.shape)
print("Start:", development_data["timestamp"].min())
print("End:", development_data["timestamp"].max())

In [ ]:
# Split the 2021 training data chronologically.
# The first 80% is used to train the models.
# The final 20% is used for validation/calibration.

split_position = int(
    len(development_data) * 0.80
)

train_model = development_data.iloc[
    :split_position
].copy()

validation_model = development_data.iloc[
    split_position:
].copy()

print("Model training observations:", len(train_model))
print("Validation observations:", len(validation_model))

print("\nTraining period:")
print(train_model["timestamp"].min())
print(train_model["timestamp"].max())

print("\nValidation period:")
print(validation_model["timestamp"].min())
print(validation_model["timestamp"].max())

#### Morning-peak prediction period

The assignment specifies the morning peak period:

**07:30–08:30**

The model can use historical observations before 07:30 to make predictions during the morning peak.

We therefore do not restrict the training dataset to only 07:30–08:30.

Instead, the morning peak is used as the main prediction/evaluation period.

In [ ]:
def select_morning_peak(data):

    time = data["timestamp"].dt.time

    return data[
        time.between(
            pd.Timestamp("07:30").time(),
            pd.Timestamp("08:30").time()
        )
    ].copy()


validation_peak = select_morning_peak(
    validation_model
)

print(
    "Validation morning-peak observations:",
    len(validation_peak)
)

display(validation_peak.head())

## 6. Model Development - diff model comparison

After establishing the spatial-information effect, we compare several regression algorithms.

The candidate models are:

- Linear Regression
- Ridge Regression
- Random Forest
- HistGradientBoosting
- Support Vector Regression
- K-Nearest Neighbours
- Multi-Layer Perceptron

The comparison will use the same training and validation data.

Hyperparameter tuning will be performed later only for the strongest candidates.

### 6.1 Model baseline to answer our questions

__Baseline model — Linear Regression__

The first model will be a simple Linear Regression.

This gives us an easy-to-understand baseline.

The purpose is not necessarily to obtain the best possible prediction.

Instead, it gives us a reference point:

> How well can we predict the future 15-minute FLOW using only the target sensor's own history and time information?

#### 6.1.1 Build the baseline model for the morning peak

In [ ]:
# Remove rows where the baseline predictors or target are missing.

baseline_data = development_data.dropna(
    subset=baseline_features + ["TARGET"]
).copy()

train_baseline = baseline_data.iloc[
    :int(len(baseline_data) * 0.80)
].copy()

validation_baseline = baseline_data.iloc[
    int(len(baseline_data) * 0.80):
].copy()

X_train = train_baseline[
    baseline_features
]

y_train = train_baseline[
    "TARGET"
]

X_validation = validation_baseline[
    baseline_features
]

y_validation = validation_baseline[
    "TARGET"
]

print("Training:", X_train.shape)
print("Validation:", X_validation.shape)

In [ ]:
# Create and train the baseline Linear Regression model.

baseline_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LinearRegression()
    )
])

baseline_model.fit(
    X_train,
    y_train
)

baseline_prediction = baseline_model.predict(
    X_validation
)

print("Baseline model trained.")

In [ ]:
# evaluation metrics

def calculate_metrics(y_true, y_pred):

    return {
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        ),
        "R2": r2_score(
            y_true,
            y_pred
        )
    }


baseline_metrics = calculate_metrics(
    y_validation,
    baseline_prediction
)

display(
    pd.DataFrame(
        [baseline_metrics],
        index=["Baseline Linear Regression"]
    )
)

In [ ]:
# Evaluate the baseline specifically during the required morning peak period.

validation_baseline_peak = select_morning_peak(
    validation_baseline
)

X_peak = validation_baseline_peak[
    baseline_features
]

y_peak = validation_baseline_peak[
    "TARGET"
]

prediction_peak = baseline_model.predict(
    X_peak
)

baseline_peak_metrics = calculate_metrics(
    y_peak,
    prediction_peak
)

display(
    pd.DataFrame(
        [baseline_peak_metrics],
        index=["Baseline - Morning Peak"]
    )
)

In [ ]:
# visualize predictions
# Compare actual and predicted FLOW for a small section of the validation period.

plot_data = validation_baseline[
    [
        "timestamp",
        "TARGET"
    ]
].copy()

plot_data["prediction"] = baseline_prediction

plot_data = plot_data.sort_values(
    "timestamp"
).head(500)

plt.figure(figsize=(14, 5))

plt.plot(
    plot_data["timestamp"],
    plot_data["TARGET"],
    label="Actual"
)

plt.plot(
    plot_data["timestamp"],
    plot_data["prediction"],
    label="Predicted"
)

plt.xlabel("Time")
plt.ylabel("Future 15-minute FLOW")
plt.title("Baseline predictions versus actual values")
plt.legend()
plt.xticks(rotation=45)
plt.show()

#### 6.1.2. Spatial experiments - our questions

We now test the main research question.

The model will initially be kept the same so that the main difference between experiments is the amount of spatial information.

We will compare:

- A: own sensor only
- B: same portal
- C: nearby portals
- D: wider network
- E: all sensors

This allows us to investigate whether additional spatial information improves prediction accuracy.\
GG: this section i understand what chat does but not the code

In [ ]:
# Get all available spatial FLOW features.

all_spatial_features = [
    column
    for column in train.columns
    if column.startswith("FLOW_SENSOR_")
    and column.endswith("_lag_1")
]

print("Number of spatial features:", len(all_spatial_features))

print(all_spatial_features[:10])

__Configuration A: Own sensor__

For each observation, the target sensor's own lagged FLOW is already represented by:

`FLOW`

and the historical lag variables such as:

`FLOW_lag_1`, `FLOW_lag_5`, `FLOW_lag_15`, etc.

This is the baseline.

__Configuration B–E: Spatial configurations__

The exact nearby and wider portal groups depend on the physical portal layout.

Once the physical portal order has been confirmed, we can construct:

- same portal,
- neighbouring portals,
- wider group,
- all portals.

For now, the all-sensor configuration can be created directly because it does not require a physical ordering assumption.

In [ ]:
# Configuration E: all sensor FLOW model information available one minute earlier.

all_sensor_features = (
    baseline_features
    + all_spatial_features
)

print(
    "Number of features in all-sensor configuration:",
    len(all_sensor_features)
)

#### 6.1.3 compare baseline with all-sensor model

In [ ]:
# Prepare data for the all-sensor model.

all_sensor_data = development_data.copy()

all_sensor_data = all_sensor_data.dropna(
    subset=all_sensor_features + ["TARGET"]
)

split_position = int(
    len(all_sensor_data) * 0.80
)

train_all = all_sensor_data.iloc[
    :split_position
]

validation_all = all_sensor_data.iloc[
    split_position:
]

X_train_all = train_all[
    all_sensor_features
]

y_train_all = train_all[
    "TARGET"
]

X_validation_all = validation_all[
    all_sensor_features
]

y_validation_all = validation_all[
    "TARGET"
]

print("Training:", X_train_all.shape)
print("Validation:", X_validation_all.shape)

### 6.2 - Models comparison
GG: there are 2 models chat took into consideration (#) we can either remove them or not

In [ ]:
models = {

    "Linear Regression": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LinearRegression()
        )
    ]),

    # "Ridge": Pipeline([
    #     (
    #         "imputer",
    #         SimpleImputer(strategy="median")
    #     ),
    #     (
    #         "scaler",
    #         StandardScaler()
    #     ),
    #     (
    #         "model",
    #         Ridge(alpha=1.0)
    #     )
    # ]),

    "Random Forest": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]),

    # "HistGradientBoosting": Pipeline([
    #     (
    #         "imputer",
    #         SimpleImputer(strategy="median")
    #     ),
    #     (
    #         "model",
    #         HistGradientBoostingRegressor(
    #             random_state=RANDOM_STATE
    #         )
    #     )
    # ]),

    "SVR": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            SVR()
        )
    ]),

    "KNN": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            KNeighborsRegressor(
                n_neighbors=5
            )
        )
    ]),

    "MLP": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            MLPRegressor(
                hidden_layer_sizes=(64, 32),
                max_iter=500,
                random_state=RANDOM_STATE
            )
        )
    ])
}

#### 6.2.1 Train all models

In [ ]:
# Train and evaluate all candidate algorithms using the baseline feature configuration first.

algorithm_results = []

for model_name, model in models.items():

    print("Training:", model_name)

    model.fit(
        X_train,
        y_train
    )

    prediction = model.predict(
        X_validation
    )

    metrics = calculate_metrics(
        y_validation,
        prediction
    )

    metrics["Model"] = model_name

    algorithm_results.append(metrics)


algorithm_results = pd.DataFrame(
    algorithm_results
)

algorithm_results = algorithm_results[
    [
        "Model",
        "MAE",
        "RMSE",
        "R2"
    ]
].sort_values("RMSE")

display(algorithm_results)

#### 6.2.2 Tuning parameters

In [ ]:
# Select the three models with the lowest validation RMSE.

finalist_models = (
    algorithm_results
    .sort_values("RMSE")
    .head(3)
)

display(finalist_models)

## 7 Model Diagnostic

Model accuracy alone does not tell us everything.

We will also investigate:

- actual versus predicted FLOW,
- prediction errors,
- errors by sensor,
- errors by portal,
- errors during the morning peak,
- unusually high/low traffic situations.

This helps us understand where the model works well and where it struggles.

### 7.1 Residuals

In [ ]:
# Use the baseline model for an initial residual analysis.

residual_data = validation_baseline[
    [
        "timestamp",
        "DP_ID",
        "PORTAL",
        "TARGET"
    ]
].copy()

residual_data["prediction"] = baseline_prediction

residual_data["error"] = (
    residual_data["TARGET"]
    - residual_data["prediction"]
)

residual_data["absolute_error"] = (
    residual_data["error"].abs()
)

display(
    residual_data.head()
)

### 7.2 Prediction error

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    residual_data["error"],
    bins=50,
    kde=True
)

plt.axvline(
    0,
    linestyle="--"
)

plt.xlabel("Prediction error")
plt.ylabel("Number of observations")
plt.title("Distribution of prediction errors")
plt.show()

### 7.3 Sensor error

In [ ]:
sensor_errors = (
    residual_data
    .groupby("DP_ID")["absolute_error"]
    .mean()
    .sort_values(ascending=False)
)

display(
    sensor_errors.to_frame(
        "MAE"
    )
)

### 7.4 Error by portal

In [ ]:
portal_errors = (
    residual_data
    .groupby("PORTAL")["absolute_error"]
    .mean()
    .sort_values(ascending=False)
)

display(
    portal_errors.to_frame(
        "MAE"
    )
)

### 7.5 Unusual traffic 

In [ ]:
# Add speed information back for diagnostic analysis.

diagnostic_data = residual_data.merge(
    train[
        [
            "timestamp",
            "DP_ID",
            "SPEED_MS_AVG"
        ]
    ],
    on=["timestamp", "DP_ID"],
    how="left"
)

display(
    diagnostic_data.head()
)

In [ ]:
# Compare prediction error with speed.

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=diagnostic_data.sample(
        min(20000, len(diagnostic_data)),
        random_state=RANDOM_STATE
    ),
    x="SPEED_MS_AVG",
    y="absolute_error",
    alpha=0.25
)

plt.xlabel("Average speed (m/s)")
plt.ylabel("Absolute prediction error")
plt.title("Prediction error under different speed conditions")
plt.show()

## 8 Train the final model

After:

1. comparing spatial configurations,
2. comparing algorithms,
3. tuning the selected model,

the final model will be trained using the complete 2021 development dataset.

Only after this step will the independent 2022 evaluation dataset be loaded.

The 2022 data will therefore remain completely unseen during model development.

In [ ]:
# Load the independent evaluation dataset.

evaluation_raw = pd.read_csv(
    FINAL_EVAL_PATH,
    sep=";"
)

print(
    "Evaluation data shape:",
    evaluation_raw.shape
)

display(
    evaluation_raw.head()
)

In [ ]:
# Create the timestamp in exactly the same way as for the
# training/development dataset.

evaluation = evaluation_raw.copy()

evaluation["timestamp"] = pd.to_datetime(
    evaluation["Date"].astype(str)
    + " "
    + evaluation["Time"].astype(str),
    errors="coerce"
)

evaluation["FLOW"] = pd.to_numeric(
    evaluation["FLOW"],
    errors="coerce"
)

evaluation["SPEED_MS_AVG"] = pd.to_numeric(
    evaluation["SPEED_MS_AVG"],
    errors="coerce"
)

evaluation = evaluation.sort_values(
    ["DP_ID", "timestamp"]
).reset_index(drop=True)

print(
    "Evaluation start:",
    evaluation["timestamp"].min()
)

print(
    "Evaluation end:",
    evaluation["timestamp"].max()
)

### Final rolling evaluation

The final evaluation follows the assignment's real-time prediction setup.

For each minute during the morning peak:

- the model receives information available at that time,
- it predicts the next 15-minute FLOW,
- the prediction window moves forward by one minute.

The evaluation is performed on the independent 2022 dataset.

In [ ]:
# Select the required final evaluation period.

evaluation_peak = select_morning_peak(
    evaluation
)

print(
    "Final evaluation observations:",
    len(evaluation_peak)
)

display(
    evaluation_peak.head()
)

## 9 Final results

The final results should report at least:

- MAE
- RMSE
- R²

for the independent 2022 evaluation period.

Results can also be reported separately for:

- each sensor,
- each portal,
- and the overall network.

In [ ]:
# This cell will be completed after the final model has been selected and the final rolling predictions have been generated.

final_results = pd.DataFrame(
    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R2"
    ]
)

display(final_results)

## Discussion

The discussion should focus on the research questions.

### RQ1
How accurately can future 15-minute traffic FLOW be predicted using only the target sensor's historical FLOW?

### RQ2
Does incorporating information from other traffic sensors improve prediction compared with using only the target sensor's historical FLOW?

### RQ3
How does prediction accuracy change as information from an increasing spatial range of sensors is incorporated?

### RQ4
Does the effect of spatial information on prediction accuracy vary between sensors and portals?

The discussion should also consider:

- differences between sensors,
- differences between portals,
- unusual traffic conditions,
- missing values and zero FLOW,
- the unknown lane correspondence,
- the limitations of using only the available sensor information,
- and the difference between validation performance and independent 2022 performance.

## Limitations

Important limitations of the analysis include:

1. Lane correspondence between sensors is unknown.

2. Spatial relationships are therefore represented using portal membership and sensor groups rather than exact lane-to-lane relationships.

3. Missing observations may reduce the number of usable rolling prediction windows.

4. Zero FLOW values are not automatically treated as errors because their meaning cannot be established from the data alone.

5. The model is evaluated during the morning peak period specified by the assignment and may not represent traffic conditions at other times of day.

6. The independent 2022 dataset is from a different period than the 2021 development data, so changes in traffic patterns may affect performance.

7. SPEED_MS_AVG is used only for exploratory and diagnostic analysis and is not used as a predictor.

## Conclusion

This project investigates short-term traffic FLOW prediction using a rolling 15-minute prediction setup.

The analysis first establishes a baseline using the target sensor's historical FLOW.

It then investigates whether adding information from other sensors improves prediction accuracy.

The spatial analysis is performed at increasing spatial extents, while avoiding assumptions about lane correspondence between sensors.

Different machine-learning algorithms are subsequently compared, and selected models are tuned using only the 2021 development data.

Finally, the selected model is evaluated on the independent 2022 dataset during the required morning peak period.

The results are used to determine whether spatial information provides useful additional information for short-term traffic FLOW prediction.